# Phase 20 — Deep-learning comparison: complete Table 14, full architecture, 10 seeds
**Resolves:** R2-12 (absolute DL AUC + 95% CI + RF AUC for ALL four datasets; architecture, embedding dimensions, optimiser,
learning rate, batch size, epochs, seeds, repeated runs). Reuses `deep_learning.py` unchanged.

Seed 42 reproduces notebook 08 exactly (AUC 0.5544 / 0.6754 / 0.7733 / 0.6906 expected; a check is printed). Seeds 42-51 give the
run-to-run variability. The paired comparison uses (a) the seed-42 model (as originally reported) and (b) the average of the ten
seeds' predicted probabilities (a seed-ensembled DL model) against the proposed RF, on identical bootstrap resamples.
**Outputs:** `S2_dl_seed_results.csv`, `S2_dl_table14_complete.csv`, `S2_dl_architecture.csv`, `S2_dl_embedding_dims.csv`
**Not tested in the authoring sandbox (no PyTorch there) — run and send any traceback.**

In [1]:
import os, sys, json, warnings
import numpy as np
import pandas as pd
warnings.filterwarnings("ignore")
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), "..", "src")))
from revision_utils import *

ENV = get_env()          # paths + config (notebooks must be run from the notebooks/ folder)
SEED = ENV["SEED"]       # 42
B = ENV["N_BOOT"]        # 2000 bootstrap resamples (config.yaml)
print("Project root:", ENV["ROOT"], "| outputs ->", ENV["REV"])

Project root: C:\maternal-mental-health-risk | outputs -> C:\maternal-mental-health-risk\reports/tables\revision


In [2]:
import torch
from torch.utils.data import DataLoader
from deep_learning import (set_seed, get_device, TabularDataset, EmbeddingMLP, prepare_tabular_arrays, train_mlp, predict_proba_mlp)
from modeling import tune_threshold, evaluate_at_threshold
DEVICE = get_device(); SEEDS = list(range(42, 52))
HP = dict(hidden_dims=(64, 32), dropout=0.3, batch_size=64, max_epochs=100, patience=10, lr=1e-3, weight_decay=1e-4)
print("device:", DEVICE, "| torch", torch.__version__, "| seeds:", SEEDS)
seed_rows, emb_rows, arch_rows, tbl14 = [], [], [], []
PUB = {"pakistan": 0.5544, "arab5_depression": 0.6754, "arab5_anxiety": 0.7733, "uganda": 0.6906}   # notebook 08 values

for tag, d in DATASETS.items():
    S = load_split(tag, ENV); num, cat, lab = d["numeric"], d["categorical"], d["label"]
    (n_tr, c_tr, n_ca, c_ca, n_te, c_te, emb_card) = prepare_tabular_arrays(S["train"], S["cal"], S["test"], num, cat)
    y_tr, y_ca, y_te = S["train"][lab].values, S["cal"][lab].values, S["test"][lab].values
    for col, (ncat, ed) in zip(cat, emb_card):
        emb_rows.append(dict(dataset=tag, variable=col, embedding_table_rows=ncat, embedding_dim=ed))
    probs_te = {}
    for seed in SEEDS:
        loader = DataLoader(TabularDataset(n_tr, c_tr, y_tr), batch_size=HP["batch_size"], shuffle=True)
        cal_ds = TabularDataset(n_ca, c_ca, y_ca)
        set_seed(seed)
        model = EmbeddingMLP(n_numeric=len(num), embedding_cardinalities=emb_card, hidden_dims=HP["hidden_dims"], dropout=HP["dropout"])
        model, best_cal_auc = train_mlp(model, loader, cal_ds, DEVICE, y_tr, seed=seed, max_epochs=HP["max_epochs"],
                                        patience=HP["patience"], lr=HP["lr"], weight_decay=HP["weight_decay"])
        p_ca = predict_proba_mlp(model, n_ca, c_ca, DEVICE); thr, _ = tune_threshold(y_ca, p_ca)
        p_te = predict_proba_mlp(model, n_te, c_te, DEVICE); probs_te[seed] = p_te
        m = evaluate_at_threshold(y_te, p_te, thr)
        seed_rows.append(dict(dataset=tag, seed=seed, best_cal_auc=best_cal_auc, threshold=thr, test_auc=m["auc_roc"],
                              accuracy=m["accuracy"], macro_f1=m["macro_f1"]))
        if seed == SEEDS[0]:
            n_params = sum(p.numel() for p in model.parameters())
            arch_rows.append(dict(dataset=tag, n_numeric_inputs=len(num), n_categorical_inputs=len(cat),
                                  total_embedding_dim=sum(e for _, e in emb_card), trainable_parameters=n_params, **HP,
                                  loss="BCEWithLogits, pos_weight = n_neg/n_pos", optimiser="Adam", batchnorm=True, activation="ReLU",
                                  early_stopping="calibration-partition AUC (best weights restored)", oversampling="none (class-weighted loss, no SMOTE)",
                                  torch_version=torch.__version__))
    sr = pd.DataFrame([r for r in seed_rows if r["dataset"] == tag])
    p_mean = np.mean([probs_te[s] for s in SEEDS], axis=0); p_rf = S["pred_test"]["rf"].values; y = S["pred_test"]["y_true"].values
    rf_a = bootstrap_ci_full(y, p_rf, B=B, seed=SEED); s42 = bootstrap_ci_full(y, probs_te[SEEDS[0]], B=B, seed=SEED)
    avg = bootstrap_ci_full(y, p_mean, B=B, seed=SEED)
    d42 = paired_bootstrap_delta(y, probs_te[SEEDS[0]], p_rf, B=B, seed=SEED); davg = paired_bootstrap_delta(y, p_mean, p_rf, B=B, seed=SEED)
    chk = sr.loc[sr.seed == SEEDS[0], "test_auc"].values[0]
    print(f"[{tag}] seed-42 AUC {chk:.4f} vs notebook-08 {PUB[tag]:.4f} -> {'REPRODUCED' if abs(chk-PUB[tag])<0.002 else 'DIFFERS (check torch/CPU)'}")
    tbl14.append(dict(dataset=tag, dl_auc_seed42=s42[0], dl_ci_lower=s42[1], dl_ci_upper=s42[2], rf_auc=rf_a[0], rf_ci_lower=rf_a[1], rf_ci_upper=rf_a[2],
                      delta_dl_minus_rf_seed42=d42["delta"], delta42_ci_lower=d42["ci_lower"], delta42_ci_upper=d42["ci_upper"],
                      dl_auc_mean_10_seeds=sr.test_auc.mean(), dl_auc_sd_10_seeds=sr.test_auc.std(ddof=1), dl_auc_min=sr.test_auc.min(), dl_auc_max=sr.test_auc.max(),
                      n_seeds_dl_above_rf=int((sr.test_auc > rf_a[0]).sum()), dl_auc_seed_averaged_probs=avg[0], avg_ci_lower=avg[1], avg_ci_upper=avg[2],
                      delta_seedavg_minus_rf=davg["delta"], deltaavg_ci_lower=davg["ci_lower"], deltaavg_ci_upper=davg["ci_upper"]))
save_table(pd.DataFrame(seed_rows), ENV, "S2_dl_seed_results.csv")
save_table(pd.DataFrame(tbl14), ENV, "S2_dl_table14_complete.csv")
save_table(pd.DataFrame(arch_rows), ENV, "S2_dl_architecture.csv")
save_table(pd.DataFrame(emb_rows), ENV, "S2_dl_embedding_dims.csv")
pd.DataFrame(tbl14).round(4)

device: cpu | torch 2.13.0+cpu | seeds: [42, 43, 44, 45, 46, 47, 48, 49, 50, 51]
[pakistan] seed-42 AUC 0.5544 vs notebook-08 0.5544 -> REPRODUCED
[arab5_depression] seed-42 AUC 0.6754 vs notebook-08 0.6754 -> REPRODUCED
[arab5_anxiety] seed-42 AUC 0.7733 vs notebook-08 0.7733 -> REPRODUCED
[uganda] seed-42 AUC 0.6906 vs notebook-08 0.6906 -> REPRODUCED
Saved: C:\maternal-mental-health-risk\reports/tables\revision\S2_dl_seed_results.csv  (40 rows)
Saved: C:\maternal-mental-health-risk\reports/tables\revision\S2_dl_table14_complete.csv  (4 rows)
Saved: C:\maternal-mental-health-risk\reports/tables\revision\S2_dl_architecture.csv  (4 rows)
Saved: C:\maternal-mental-health-risk\reports/tables\revision\S2_dl_embedding_dims.csv  (43 rows)


,dataset,dl_auc_seed42,dl_ci_lower,dl_ci_upper,rf_auc,rf_ci_lower,rf_ci_upper,delta_dl_minus_rf_seed42,delta42_ci_lower,delta42_ci_upper,...,dl_auc_sd_10_seeds,dl_auc_min,dl_auc_max,n_seeds_dl_above_rf,dl_auc_seed_averaged_probs,avg_ci_lower,avg_ci_upper,delta_seedavg_minus_rf,deltaavg_ci_lower,deltaavg_ci_upper
0,pakistan,0.5544,0.5315,0.5767,0.5571,0.5353,0.5792,-0.0027,-0.0186,0.0133,...,0.0065,0.5484,0.5716,6,0.5655,0.5434,0.5881,0.0084,-0.0053,0.0222
1,arab5_depression,0.6754,0.6155,0.7357,0.6867,0.6252,0.7468,-0.0113,-0.0395,0.0168,...,0.0130,0.6503,0.6951,1,0.6831,0.6219,0.7444,-0.0036,-0.0240,0.0164
2,arab5_anxiety,0.7733,0.6861,0.8519,0.7815,0.6958,0.8631,-0.0082,-0.0417,0.0270,...,0.0144,0.7733,0.8208,8,0.8015,0.7207,0.8772,0.0200,-0.0039,0.0448
3,uganda,0.6906,0.6708,0.7102,0.6965,0.6770,0.7162,-0.0059,-0.0142,0.0020,...,0.0015,0.6893,0.6935,0,0.6961,0.6763,0.7156,-0.0004,-0.0067,0.0058


**Limitation to state:** DL early stopping uses the same calibration partition later used for threshold selection; test AUC is
unaffected, but calibration-partition AUCs for DL are optimistic. Several Arab-region columns are ordinal codes
(e.g. `WomenAge`, `MariageAge`) treated as numeric inputs, as in the tree models.